# Probabilistic Sharpe Ratio (PSR)

Bailey & López de Prado, 2012; MLAM ch. 8

A Sharpe ratio is an estimate, so it has an error. The PSR turns it into a probability: given this track record, how likely is it that the true Sharpe is above a threshold SR*?

PSR(SR*) = Φ( (SR̂ − SR*) · √(T − 1) / √(1 − γ₃ · SR̂ + (γ₄ − 1)/4 · SR̂²) )

The PSR is calculated as follows:
- Excess returns
- Daily Sharpe
- Treshold in daily returns
- Apply the PSR formula
- From z to a probability

In [27]:
# Imports

import pandas as pd
import numpy as np
from vportfolio.params import TRIALS_LOG
from vportfolio import load_data, trials
from scipy.stats import skew, kurtosis, norm

In [21]:
# calculate the PSR on given inputs
# will return a probability between 0 and 1

def psr(nav, rf_daily, sr_threshold = 0):
    # calculate daily returns
    daily_returns = nav.pct_change(fill_method=None).dropna()
    # allign the indices of returns and risk free rate, and fill NAs with 0s
    rf_alligned = rf_daily.reindex(daily_returns.index).fillna(0)
    daily_excess_returns = daily_returns - rf_alligned
    # caluclate daily Sharpe
    d_sharpe = daily_excess_returns.mean() / daily_excess_returns.std()
    # define additional variables (ingredients of uncertainty)
    T = len(daily_excess_returns)
    skew_ = skew(daily_excess_returns)
    kurt_ = kurtosis(daily_excess_returns, fisher=False) 
    # convert annual treshold to daily
    d_tr = sr_threshold / np.sqrt(252)
    # PSR formula
    z = (d_sharpe - d_tr)*np.sqrt(T-1)/np.sqrt(1-skew_*d_sharpe+(kurt_-1)/4*d_sharpe**2)
    # turn z to a probability between 0 and 1
    return norm.cdf(z)

In [22]:
# function testing

rng = np.random.default_rng(0)
dates = pd.bdate_range("2020-01-01", periods=2520) # 10 years of business days
test_rf = pd.Series(0.0, index=dates) # risk-free rate of 0
# a random NAV: daily returns drawn with mean 0.0004 and std 0.01 (the sample comes out weaker)
returns = pd.Series(rng.normal(0.0004, 0.01, size=2520), index=dates)
nav = (1 + returns).cumprod()
# a "strong" NAV: draws rescaled so the sample has exactly mean 0.0004 and std 0.01 (daily Sharpe 0.04)
draws = rng.normal(size=2520)
strong = pd.Series((draws - draws.mean()) / draws.std() * 0.01 + 0.0004, index=dates)
nav_strong = (1 + strong).cumprod()
# 1. always true: the PSR is a probability, between 0 and 1
test = psr(nav, test_rf)
assert 0 <= test <= 1
# 2. known answers
# 2a. threshold equal to the strategy's own (annual) Sharpe → exactly 0.5
nav_returns = nav.pct_change(fill_method=None).dropna() # the same returns psr uses
own_sharpe = nav_returns.mean() / nav_returns.std() * np.sqrt(252)
test2 = psr(nav, test_rf, own_sharpe)
assert np.isclose(test2, 0.5, atol=1e-6)
# 2b. a strong Sharpe over 10 years → very likely above 0
assert psr(nav_strong, test_rf) > 0.95
# 3. inputs matter
# 3a. a higher threshold is harder to beat → lower PSR
assert test2 < test
# 3b. a shorter track record (100 days) with the same strength → lower PSR
assert psr(nav_strong.iloc[:100], test_rf) < psr(nav_strong, test_rf)

print("psr passed all tests")

psr passed all tests


Now that we have a PSR function, let's run it on all NAVs

In [29]:
# import the data

log = pd.read_csv(TRIALS_LOG).set_index("name")
navs = trials.load_navs(log.index.tolist())
shared_data = load_data._load_shared()
rf_daily = shared_data["rf_daily"]

In [37]:
# run the PSR on each strategy

results = {} # empty dictionary to populate with the individual dictionaries

trs_sixty_forty = log.loc["sixty_forty", "sharpe"] # set the second treshold to sixty forty
trs_equal_weights = log.loc["equal_weights", "sharpe"] # set the third treshold to equal weights

for name in navs.columns: # build a dictionary with each nav's three sharpes
    nav = navs[name].dropna()
    results[name] = {
        "vs_0" : psr(nav, rf_daily, sr_threshold = 0),
        "vs_6040" : psr(nav, rf_daily, sr_threshold = trs_sixty_forty),
        "vs_ew" : psr(nav, rf_daily, sr_threshold = trs_equal_weights)
    }
psr_table = pd.DataFrame(results).T

In [42]:
psr_table.round(2)

,vs_0,vs_6040,vs_ew
sixty_forty,1.00,0.50,0.74
equal_weights,0.98,0.26,0.50
risk_weighted,0.99,0.40,0.65
hrp,0.99,0.31,0.56
nco,0.97,0.23,0.46
min_var_raw,0.96,0.19,0.41
min_var_denoised,0.97,0.23,0.46


In [45]:
print(psr_table.round(2).to_markdown())

|                  |   vs_0 |   vs_6040 |   vs_ew |
|:-----------------|-------:|----------:|--------:|
| sixty_forty      |   1    |      0.5  |    0.74 |
| equal_weights    |   0.98 |      0.26 |    0.5  |
| risk_weighted    |   0.99 |      0.4  |    0.65 |
| hrp              |   0.99 |      0.31 |    0.56 |
| nco              |   0.97 |      0.23 |    0.46 |
| min_var_raw      |   0.96 |      0.19 |    0.41 |
| min_var_denoised |   0.97 |      0.23 |    0.46 |


1. Against 0: every strategy’s Sharpe is genuinely positive (0.96–1.00). All seven earn a real excess return over T-bills. Over 17 years, that’s not luck
2. Against 60/40: no candidate is likely to beat it. The probability that the candidate truly beats 60/40 is only 19–31%. That’s lower than a coin flip in every case.
Even risk-weighted, the strongest non-benchmark, reaches only 40%
3. Against equal weight: the candidates are indistinguishable from it
    - HRP: 56%, barely more than a coin flip
    - NCO and min_var_denoised: 46%,
    - min_var_raw: 41%, a little below a coin flip

**Conclusion**: every strategy’s Sharpe is very likely above 0 (0.96–1.00). Against equal weight’s Sharpe, the candidates are indistinguishable from a coin flip (0.41–0.56; HRP highest). Against 60/40’s Sharpe, the probability that any candidate truly beats it is only 0.19–0.31. The candidates offer lower risk, not a demonstrably better risk-adjusted return